Φόρτωση Μοντέλου: facebook/nllb-200-distilled-600M

In [1]:
!pip install -q -U transformers

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang = "ell_Grek")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Δοκιμή

(https://huggingface.co/docs/transformers/model_doc/nllb)

In [4]:
text = "Καλημέρα, σήμερα είναι Σάββατο και ο καιρός είναι ζεστός."
inputs = tokenizer(text, return_tensors = "pt").to(model.device)

translated_tokens = model.generate(**inputs, forced_bos_token_id = tokenizer.convert_tokens_to_ids("eng_Latn"), max_length=30)
tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

"Good morning, it's Saturday and the weather is hot."

Εκτέλεση στο Flores Dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json

In [7]:
# Φόρτωση των δεδομένων

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_ell.json", "r", encoding = "utf-8") as f:
  ell_dataset = json.load(f)

with open("/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1/Flores_eng.json", "r", encoding = "utf-8") as f:
  eng_dataset = json.load(f)

In [8]:
len(ell_dataset), len(eng_dataset)

(1012, 1012)

In [9]:
# Τρέχουμε στο ell_dataset

batch_size = 32
model_translations = []

In [10]:
for i in range(0, len(ell_dataset), batch_size):
  batch = ell_dataset[i : i + batch_size]

  inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(model.device)

  all_translations = model.generate(**inputs,forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),max_length=256)

  batch_preds = tokenizer.batch_decode(all_translations, skip_special_tokens=True)
  model_translations.extend(batch_preds)

In [11]:
len(model_translations)

1012

Αποθήκευση των μεταφράσεων του μοντέλου

In [12]:
import os

In [13]:
filename = "facebook_nllb_200_translations" + ".json"
path = os.path.join(f"/content/drive/MyDrive/Μηχανική Μετάφραση_Εργασία 1", filename)

with open(path, "w", encoding="utf-8") as f:
    json.dump(model_translations, f, ensure_ascii=False, indent=4)

Αξιολόγηση Μοντέλου

In [14]:
!pip install -q -q evaluate sacrebleu

In [15]:
import evaluate

In [ ]:
BLEU = evaluate.load("sacrebleu")
CHRF = evaluate.load("chrf")

BLEU_results = BLEU.compute(predictions = model_translations, references = [[ref] for ref in eng_dataset])
CHRF_results = CHRF.compute(predictions = model_translations, references = [[ref] for ref in eng_dataset])

print("BLEU:", BLEU_results["score"])
print("chrF:", CHRF_results["score"])

BLEU: 33.96742699443383
chrF: 59.75878770624221
